# Ijeoma AI — test pages on a Colab GPU

Starts the three Ijeoma AI test pages (Workbench, Chat, Compare) on a Colab GPU
and prints a public **gradio.live** link you can open on any device or share.

**Before you run:**
1. **Runtime → Change runtime type → T4 GPU**
2. Colab **🔑 Secrets**: `HF_TOKEN` with notebook access on (the same token used for training)
3. The code: either set `REPO_URL` below to the GitHub repository, or put
   `ijeoma_app_bundle.zip` in the **Ijeoma** folder in your Drive.

Then **Runtime → Run all**. The link appears at the bottom after ~3–5 minutes
and lasts up to 72 hours, or until this Colab session stops.

In [ ]:
# Keras reads the backend when first imported, so this must run first.
import os, sys
if "keras" in sys.modules:
    raise RuntimeError("Keras already imported - Runtime > Restart session, then run from the top.")
os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.90"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU - Runtime > Change runtime type > T4"

In [ ]:
!pip install -q -U keras keras-hub "gradio==5.49.1"

In [ ]:
from google.colab import userdata
for name in ("HF_TOKEN", "IjeomaHF", "HUGGINGFACE_TOKEN"):
    try:
        tok = userdata.get(name)
    except Exception:
        tok = None
    if tok:
        os.environ["HF_TOKEN"] = tok
        print("Hugging Face token found in secret", name)
        break
else:
    raise RuntimeError("No Hugging Face token in Colab Secrets. Add HF_TOKEN and enable notebook access.")

In [ ]:
# Get the code: from GitHub if REPO_URL is set, otherwise from the bundle on Drive.
REPO_URL = ""   # e.g. "https://github.com/<owner>/ijeoma-ai"
APP_DIR = "/content/ijeoma-ai"

if REPO_URL:
    !rm -rf {APP_DIR} && git clone -q --depth 1 {REPO_URL} {APP_DIR}
else:
    from google.colab import drive
    drive.mount("/content/drive")
    import glob
    hits = glob.glob("/content/drive/MyDrive/Ijeoma/ijeoma_app_bundle.zip") + \
           glob.glob("/content/drive/MyDrive/*/Ijeoma/ijeoma_app_bundle.zip")
    if not hits:
        raise FileNotFoundError("Set REPO_URL, or put ijeoma_app_bundle.zip in the Ijeoma folder on Drive.")
    !rm -rf {APP_DIR} && mkdir -p {APP_DIR} && unzip -q -o "{hits[0]}" -d {APP_DIR}

ADAPTER = f"{APP_DIR}/models/ijeoma_lora_gemma-3-1b-it-r8-lr0.0002-e1.npz"
print("adapter present:", os.path.exists(ADAPTER))

## Start the pages

The public link is printed below as `Running on public URL: https://….gradio.live`.
Pages: `/` Workbench · `/chat` Chat · `/compare` Compare.
This cell keeps running while the pages are up; stop it to shut them down.

In [ ]:
!cd {APP_DIR} && python app/ijeoma_app.py --adapter {ADAPTER} --share --preload-base